# STEPL0 — Missive-receipt cue distribution over time

Cue inventory for Track L: **open** (*ontfangen een missive/brief…*) × **close** (*geen resolutie gevallen / vereischende*).

- Corpus: `resolutions_flat` via `data_io`
- Dates: ISO strings → year/month by slice (no `pd.to_datetime`; corpus includes pre-1678)
- Heavy cell loads full text once; re-run only when cues change


In [ ]:
import re

import matplotlib.pyplot as plt
import pandas as pd
from data_io import load, resolve

CLOSE = re.compile(
    r"geen\s+resolutie\s+(?:is\s+)?gevallen|"
    r"geen\s+resolutie\s+vere[iy]sch",
    re.I,
)
OPEN = re.compile(
    r"ontfangen\s+een(?:en)?\s+(?:missive|missiue|brief|brieff)",
    re.I,
)

seeds = load("missive_receipt_seeds")
print("seeds", len(seeds))
print("flat", resolve("resolutions_flat"))

## Score cues (once)

Loads `id`, `date`, `resolutions_text` only. Expect ~10–60s depending on disk.


In [ ]:
df = pd.read_parquet(
    resolve("resolutions_flat"),
    columns=["id", "type", "date", "resolutions_text"],
)
date_s = df["date"].astype(str)
df = df.assign(
    year=pd.to_numeric(date_s.str[:4], errors="coerce"),
    month=pd.to_numeric(date_s.str[5:7], errors="coerce"),
    record_type=df["type"].astype(str),
)
text = df["resolutions_text"].fillna("").astype(str)
df = df.assign(
    open_hit=text.str.contains(OPEN),
    close_hit=text.str.contains(CLOSE),
)
df = df.assign(both=df["open_hit"] & df["close_hit"])
resolution_type = pd.Series("other", index=df.index, dtype="string")
resolution_type = resolution_type.mask(df["close_hit"], "missive_close_only")
resolution_type = resolution_type.mask(df["open_hit"], "missive_open")
resolution_type = resolution_type.mask(df["both"], "missive_receipt")
df = df.assign(resolution_type=resolution_type)
# Keep text only for later spot-checks; cue is the work frame
cue = df[
    [
        "id",
        "record_type",
        "resolution_type",
        "year",
        "month",
        "open_hit",
        "close_hit",
        "both",
    ]
].copy()
print(
    f"n={len(cue):,} open={cue['open_hit'].sum():,} "
    f"close={cue['close_hit'].sum():,} both={cue['both'].sum():,}"
)
print("year range", int(cue["year"].min()), "–", int(cue["year"].max()))
print("resolution_type:")
print(cue["resolution_type"].value_counts())


## By year

Absolute counts and share of resolutions that year (`both / n`).


In [ ]:
by_year = (
    cue.groupby("year", dropna=True)
    .agg(
        n=("id", "size"),
        open=("open_hit", "sum"),
        close=("close_hit", "sum"),
        both=("both", "sum"),
    )
    .astype({"open": int, "close": int, "both": int})
    .assign(both_share=lambda d: d["both"] / d["n"])
)
by_year.tail(10)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
by_year[["open", "close", "both"]].plot(ax=axes[0], lw=1)
axes[0].set_ylabel("count")
axes[0].set_title("Missive cues by year")
axes[0].legend(loc="upper left")
by_year["both_share"].plot(ax=axes[1], color="C2", lw=1)
axes[1].set_ylabel("both / n")
axes[1].set_xlabel("year")
axes[1].set_title("Share of resolutions with open+close")
fig.tight_layout()

## By month (all years pooled)

Seasonality of `both` vs baseline volume.


In [ ]:
by_month = (
    cue.dropna(subset=["month"])
    .groupby("month")
    .agg(n=("id", "size"), both=("both", "sum"), open=("open_hit", "sum"))
    .astype(int)
    .assign(both_share=lambda d: d["both"] / d["n"])
)
by_month

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
by_month[["n", "both"]].plot.bar(ax=axes[0], rot=0)
axes[0].set_title("Volume vs both (all years)")
axes[0].set_xlabel("month")
by_month["both_share"].plot.bar(ax=axes[1], color="C2", rot=0)
axes[1].set_title("both / n by month")
axes[1].set_xlabel("month")
fig.tight_layout()

## One year detail + year×month heatmap

Default focal year `1780` (STEPL0 POC). Change `FOCAL_YEAR` as needed.


In [ ]:
FOCAL_YEAR = 1780

focal = cue.loc[cue["year"] == FOCAL_YEAR]
focal_month = (
    focal.groupby("month")
    .agg(n=("id", "size"), open=("open_hit", "sum"), close=("close_hit", "sum"), both=("both", "sum"))
    .astype(int)
)
print(f"year {FOCAL_YEAR}")
display(focal_month)

heat = (
    cue.dropna(subset=["year", "month"])
    .loc[lambda d: (d["year"] >= 1700) & (d["year"] <= 1795)]
    .pivot_table(index="year", columns="month", values="both", aggfunc="sum", fill_value=0)
)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(heat.to_numpy(), aspect="auto", cmap="YlOrRd", interpolation="nearest")
ax.set_yticks(range(0, len(heat), 5))
ax.set_yticklabels(heat.index[::5].astype(int))
ax.set_xticks(range(12))
ax.set_xticklabels(range(1, 13))
ax.set_xlabel("month")
ax.set_ylabel("year")
ax.set_title("both-hit counts (1700–1795)")
fig.colorbar(im, ax=ax, shrink=0.6, label="both")
fig.tight_layout()

## Spot-check samples

Open-only vs both for the focal year (sanity: open-only often aanstelling / referral).


In [ ]:
FOCAL_YEAR  # from earlier cell

def _sample_ids(mask, n=5):
    return cue.loc[mask, "id"].head(n).tolist()

open_only = _sample_ids((cue["year"] == FOCAL_YEAR) & cue["open_hit"] & ~cue["both"])
both_ids = _sample_ids((cue["year"] == FOCAL_YEAR) & cue["both"])
print("open-only", open_only)
print("both", both_ids)

want = set(open_only + both_ids)
peek = df.loc[df["id"].isin(want), ["id", "resolutions_text"]]
for _, row in peek.iterrows():
    tag = "both" if row["id"] in both_ids else "open-only"
    print(f"\n--- {tag} {row['id']} ---")
    print(str(row["resolutions_text"])[:400])